# Notebook17 shared input pipeline
Purpose: preserve the original sample transformations while measuring ordered CPU preparation with0,4,8 threads. Inputs: audited TRAIN membership/config. Outputs: exact-equivalence and TRAIN-only timing receipts. Importing tagged definitions does not run benchmarks or training. Runtime: definitions under5s; invoked selection roughly1–3min. No checkpoint-bearing state exists; sampler cursors remain owned by the training checkpoints.

In [ ]:
"""Shared ordered-input, source, and metadata helpers for Notebook17. Import never trains."""
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
import copy,hashlib,json,math,random,sys,time,types
import nbformat,numpy as np,torch
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()

def export_hash(path):
    return hashlib.sha256('\n'.join(c.source for c in nbformat.read(path,4).cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])).encode()).hexdigest()

def load(root,relative,name):
    root=Path(root).resolve();path=(root/relative).resolve()
    if not path.is_relative_to(root) or path.suffix!='.ipynb':raise ValueError('Tagged notebook path must stay inside project')
    if name in sys.modules:
        module=sys.modules[name]
        if Path(module.__file__).resolve()!=path:raise ValueError('Module path changed')
        return module
    module=types.ModuleType(name);module.__file__=str(path);sys.modules[name]=module
    cells=[c for c in nbformat.read(path,4).cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])]
    if not cells:raise ValueError('No explicit tagged definitions')
    try:
        for i,c in enumerate(cells):exec(compile(c.source,str(path)+'#ahr-export-'+str(i),'exec'),module.__dict__)
    except BaseException:sys.modules.pop(name,None);raise
    return module

def bootstrap(root):
    module=load(root,'fmm_jepa/bootstrap.ipynb','fmm17_bootstrap');module.install_notebook_package(Path(root));return module

def modules(root):
    bootstrap(root)
    return (load(root,'experiments/15_vit_geometry/runner.ipynb','fmm17_isolated_runner15'),
        load(root,'experiments/15_vit_geometry/encoder.ipynb','fmm15_encoder'),
        load(root,'experiments/17_fixed_kernel/operator.ipynb','fmm17_operator'))

def source_hashes(root):
    root=Path(root).resolve();bootstrap(root)
    runner=load(root,'experiments/15_vit_geometry/runner.ipynb','fmm17_isolated_runner15')
    # Start from the unbound foundation identity; later bindings cannot leak old16 operator definitions.
    result=runner._BASE_ORIGINAL_HASHES(root)
    for folder in ('experiments/11_manifold_long','experiments/15_vit_geometry','experiments/16_hierarchical_confirmation','experiments/17_fixed_kernel'):
        for path in sorted((root/folder).glob('*.ipynb')):result[str(path.relative_to(root))]=export_hash(path)
    return result

def canonical(value):return json.loads(json.dumps(value,allow_nan=False))
def digest(value):return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def file_record(path):
    path=Path(path);return {'path':str(path.resolve()),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
def read(path):return json.loads(Path(path).read_text())
def atomic_json(path,value):
    bootstrap(ROOT);from fmm_jepa.training import atomic_json as write;return write(path,canonical(value))
def immutable(path,value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if read(path)!=value:raise ValueError('Immutable registration changed:'+str(path))
    else:atomic_json(path,value)
    return value

def check_deadline(deadline,required=0.):
    if deadline is not None and time.time()+float(required)>=float(deadline):raise TimeoutError('Original17 deadline insufficient/expired; UNKNOWN, never reset')

def environment(config):
    if os.environ.get('CUBLAS_WORKSPACE_CONFIG')!=':4096:8':raise RuntimeError('Deterministic CUDA workspace required')
    torch.set_num_threads(2);torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    device=str(config.get('device','cuda:0'))
    if device.startswith('cuda'):
        if not torch.cuda.is_available():raise RuntimeError('Registered GPU unavailable')
        torch.cuda.set_device(torch.device(device))
    return {'device':device,'dtype':'float32','tf32':False,'deterministic':True,'cpu_threads':2}

def _original_providers(root=ROOT):
    bootstrap(root)
    import fmm_jepa.food101_data as data
    import fmm_jepa.food101_evaluation as evaluation
    old=getattr(data,'_fmm11_input_pipeline',None)
    return data,evaluation,(old['original'] if old else data.make_batch),(old['original_evaluation_batch'] if old else evaluation._batch)

def install_input_pipeline(workers=0,root=ROOT):
    """Switch an idle process among 0/4/8 workers; no prefetch/sampler/background state.

    Existing notebook11 original-example formulas are called synchronously and in order.
    Every thread joins before return. Switches are only legal outside a provider call.
    Scientific plans freeze one selected worker count for both architectures.
    """
    if isinstance(workers,bool) or int(workers)!=workers or int(workers) not in (0,4,8):raise ValueError('Registered workers must be0,4,8')
    workers=int(workers);data,evaluation,original,original_eval=_original_providers(root)
    data.make_batch=original;evaluation._batch=original_eval
    if hasattr(data,'_fmm11_input_pipeline'):delattr(data,'_fmm11_input_pipeline')
    if workers:
        source=load(root,'experiments/11_manifold_long/input_pipeline.ipynb','fmm17_original_pipeline11')
        result=source.install_input_pipeline(workers)
    else:result={'status':'SERIAL','workers':0,'ordered':True,'prefetch':False,'all_futures_join_before_return':True}
    result={**result,'source_notebook':'experiments/11_manifold_long/input_pipeline.ipynb','worker_selection':'Frozen equally for ViT and candidate'}
    return result

def exact_tree(left,right,path='root'):
    if isinstance(left,torch.Tensor):
        if not isinstance(right,torch.Tensor) or left.dtype!=right.dtype or left.shape!=right.shape or not torch.equal(left,right):raise AssertionError(path)
    elif isinstance(left,np.ndarray):
        if not isinstance(right,np.ndarray) or not np.array_equal(left,right):raise AssertionError(path)
    elif isinstance(left,dict):
        if not isinstance(right,dict) or set(left)!=set(right):raise AssertionError(path)
        for k in left:exact_tree(left[k],right[k],path+'.'+str(k))
    elif isinstance(left,(list,tuple)):
        if type(left)!=type(right) or len(left)!=len(right):raise AssertionError(path)
        for i,(a,b) in enumerate(zip(left,right)):exact_tree(a,b,path+'.'+str(i))
    elif left!=right:raise AssertionError((path,left,right))

def rng_snapshot():return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state().clone()}

def verify_pipeline(bundle,config,workers=(0,4,8),deadline=None):
    """Bitwise images/masks/metadata, local RNG independence, sampler exact replay; TRAIN images only."""
    check_deadline(deadline);bootstrap(ROOT)
    from fmm_jepa.food101_training import ConsumedSampler
    from fmm_jepa.food101_data import EvaluationDataset
    data,evaluation,original,original_eval=_original_providers()
    count=min(int(config['training']['microbatch']),len(bundle['train_ids']));ids=list(bundle['train_ids'])[:count]
    if not ids:raise ValueError('Nonempty audited training membership required')
    ids=ids+[ids[0]];occurrences=list(range(len(ids)));before=rng_snapshot();reference=original(bundle['dataset'],ids,occurrences,config,training=True)
    eval_dataset=EvaluationDataset(bundle['dataset'],ids,config);eval_reference=original_eval(eval_dataset,0,len(ids))
    exact_tree(before,rng_snapshot(),'reference_global_rng');cases=[]
    for workers_count in workers:
        check_deadline(deadline);install_input_pipeline(workers_count)
        actual=data.make_batch(bundle['dataset'],ids,occurrences,config,training=True)
        exact_tree(reference,actual,'training'+str(workers_count));exact_tree(before,rng_snapshot(),'global_rng')
        exact_tree(eval_reference,evaluation._batch(eval_dataset,0,len(ids)),'train_only_evaluation'+str(workers_count));exact_tree(before,rng_snapshot(),'evaluation_rng')
        sampler=ConsumedSampler(list(bundle['train_ids'])[:max(count+3,7)],seed=int(config['seeds']['data_order']));sampler.take(3);saved=sampler.state_dict()
        replay_ids,replay_occ=sampler.take(count);a=data.make_batch(bundle['dataset'],replay_ids,replay_occ,config)
        restored=ConsumedSampler(list(bundle['train_ids'])[:max(count+3,7)],seed=int(config['seeds']['data_order']));restored.load_state_dict(saved)
        second_ids,second_occ=restored.take(count);b=data.make_batch(bundle['dataset'],second_ids,second_occ,config)
        exact_tree(a,b,'sampler_replay');exact_tree(sampler.state_dict(),restored.state_dict(),'sampler_cursor')
        cases.append({'workers':workers_count,'status':'PASS','bitwise_images_labels_augmentations_masks_padding_metadata':True,'exact_sampler_replay':True,'global_rng_unchanged':True,'train_only_evaluation_images_labels_ids_bitwise':True})
    install_input_pipeline(0)
    return {'schema':'notebook17-input-equivalence/v1','status':'PASS','cases':cases,'train_only':True,'official_test_pixels_read':0,'source':file_record(ROOT/'experiments/11_manifold_long/input_pipeline.ipynb')}

def benchmark_input_pipeline(bundle,config,deadline,workers=(0,4,8),repeats=3):
    """Actual physical microbatches repeated for one effective batch, CPU preparation only."""
    check_deadline(deadline);environment(config)
    if isinstance(repeats,bool) or int(repeats)!=repeats or not 1<=int(repeats)<=10:raise ValueError('Bounded positive1..10 timing repeats required')
    proof=verify_pipeline(bundle,config,workers,deadline)
    data,_,_,_=_original_providers();batch=int(config['training']['microbatch']);accum=int(config['training']['accumulation_steps']);need=batch*accum
    ids=(list(bundle['train_ids'])*math.ceil(need/len(bundle['train_ids'])))[:need];occurrences=[4]*need;rows=[]
    for count in workers:
        check_deadline(deadline);install_input_pipeline(count)
        data.make_batch(bundle['dataset'],ids[:batch],occurrences[:batch],config)
        timings=[]
        for rep in range(int(repeats)):
            check_deadline(deadline);started=time.perf_counter()
            for i in range(accum):data.make_batch(bundle['dataset'],ids[i*batch:(i+1)*batch],occurrences[i*batch:(i+1)*batch],config)
            timings.append(time.perf_counter()-started)
        rows.append({'workers':int(count),'physical_batch':batch,'effective_batch':need,'seconds':timings,'median_seconds':float(np.median(timings))})
    chosen=min(rows,key=lambda r:(r['median_seconds'],r['workers']));install_input_pipeline(chosen['workers'])
    return {'schema':'notebook17-input-selection/v1','status':'PASS','selected_workers':chosen['workers'],'rows':rows,'verification':proof,
        'selection':'Lowest median TRAIN-only CPU preparation time; ties favor fewer workers; same frozen count for both models',
        'speedup_vs_serial':rows[0]['median_seconds']/chosen['median_seconds'],'complete_jepa_speedup':'UNMEASURED','train_only':True,'official_test_pixels_read':0}

def learned_source(root,variant='vit',source_run='vit-geometry-001'):
    """First registered seed; no performance selection and no official test pixel reads."""
    root=Path(root);bootstrap(root);runner=load(root,'experiments/15_vit_geometry/runner.ipynb','fmm17_isolated_runner15')
    folder=root/'artifacts/notebook_15'/source_run;handoff=read(folder/'handoff_15_to_16.json')
    if handoff.get('status')!='COMPLETE' or handoff.get('fixture_source'):raise ValueError('Completed scientific15 handoff required')
    plan_path=handoff['candidate_plan'];plan_path=plan_path['path'] if isinstance(plan_path,dict) else plan_path
    training_path=handoff['training_result'];training_path=training_path['path'] if isinstance(training_path,dict) else training_path
    if file_record(plan_path)['sha256']!=handoff['plan_sha256'] or file_record(training_path)['sha256']!=handoff['training_sha256']:raise ValueError('Frozen15 plan/training document checksum mismatch')
    plan=read(plan_path);training=read(training_path);seed=sorted(plan['seeds'])[0]
    arm=next(a for a in plan['arms'] if int(a['seed'])==seed and a['variant']==variant);record=training['arms'][arm['id']]['milestones'][str(plan['updates'])]
    if file_record(record['path'])['sha256']!=record['sha256']:raise ValueError('Frozen teacher checksum mismatch')
    config=copy.deepcopy(arm['config']);payload=torch.load(record['path'],map_location='cpu',weights_only=False)
    model=runner._default_build_model(config,arm['kind']).cpu();model.load_state_dict(payload['model'])
    return model,config,{'role':'FROZEN_NOTEBOOK15_TRAINING_ONLY','seed':seed,'variant':variant,'checkpoint':record,'plan':file_record(plan_path),'training':file_record(training_path),'official_test_pixels_read':0}
